# 04 · Modelo LightGBM y experimentos (exploración)

Clasificador LightGBM validado con `GroupKFold` por vehículo (ningún vehículo en train y test a la vez), y la serie de
experimentos que guió las decisiones: fugas por nulos, temperatura como calendario, país contra altitud (con restricción
monotónica), punto de operación mes a mes, SHAP y chequeos de cohorte.

**Entradas:** `salidas/dataset.parquet`, `ciudades_coordenadas.csv` (GeoNames + 10 ciudades cargadas a mano) ·
**Salidas:** `salidas/altitud_ciudades.csv` (API de elevación de Open-Meteo), `salidas/scores_dashboard.*`.

Resultado final de esta exploración (modelo v3: comportamiento + ralentí + mecanismo + altitud monotónica):
PR-AUC 0,176 (azar 0,028); con umbral 98 %, 5,4 % de meses de sanos con alerta y 49 % de fallas detectadas en los 30 días
previos. Los números varían ±0,01 entre máquinas: diferencias menores no son concluyentes.

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score
import lightgbm as lgb

SALIDAS = "salidas/"
DATOS = "../../Datasets/"
df = pd.read_parquet(SALIDAS + "dataset.parquet")
df.shape

In [ ]:
cols = ["acum_media_30d", "pct_lleno_30d", "dist_regen_30d", "consumo_30d", "aceite_30d"]
df.groupby("failed")[cols].apply(lambda g: g.isna().mean()).round(3)

,acum_media_30d,pct_lleno_30d,dist_regen_30d,consumo_30d,aceite_30d
failed,,,,,
0,0.003,0.003,0.288,0.007,0.010
1,0.000,0.000,0.120,0.009,0.009


In [ ]:
features = ["dias_activos_30d", "km_30d", "viajes_30d", "pct_cortos_30d", "pct_motor_frio_30d",
            "vel_media_30d", "temp_aire_30d", "consumo_30d", "aceite_30d",
            "pct_lleno_14d", "pct_lleno_30d", "acum_media_30d", "acum_max_30d", "viajes_lleno_30d",
            "regeneraciones_30d", "dist_regen_30d", "limpiezas_cortadas_30d",
            "tend_lleno", "tend_consumo", "tend_dist_regen", "min_ralenti_30d", "ralentis_largos_30d",
               "viajes_filtro_cargado_30d", "cortos_filtro_cargado_30d",
               "regen_interrumpidas_30d", "regen_completas_30d", "tasa_regen_exitosa_30d"]

X = df[features]
y = df["target"]
grupos = df["VehicleCode"]
print(X.shape, "positivos:", round(y.mean(), 4))

(207921, 27) positivos: 0.0275


In [ ]:
print("PR-AUC al azar:          ", round(y.mean(), 4))
print("PR-AUC regla pct_lleno:  ", round(average_precision_score(y, df["pct_lleno_30d"].fillna(0)), 4))

PR-AUC al azar:           0.0275
PR-AUC regla pct_lleno:   0.0424


In [ ]:
cv = GroupKFold(n_splits=5)
oof = np.zeros(len(df))

for k, (tr, te) in enumerate(cv.split(X, y, grupos)):
    peso = (y.iloc[tr] == 0).sum() / (y.iloc[tr] == 1).sum()
    modelo = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                min_child_samples=100, subsample=0.8, subsample_freq=1,
                                colsample_bytree=0.8, scale_pos_weight=peso, verbose=-1)
    modelo.fit(X.iloc[tr], y.iloc[tr])
    oof[te] = modelo.predict_proba(X.iloc[te])[:, 1]
    print(f"fold {k}: PR-AUC {average_precision_score(y.iloc[te], oof[te]):.4f}")

df["score"] = oof
print("PR-AUC total:", round(average_precision_score(y, oof), 4),
      "| ROC-AUC:", round(roc_auc_score(y, oof), 4))

fold 0: PR-AUC 0.1327
fold 1: PR-AUC 0.1576
fold 2: PR-AUC 0.1904
fold 3: PR-AUC 0.2093
fold 4: PR-AUC 0.1637
PR-AUC total: 0.1657 | ROC-AUC: 0.7461


In [ ]:
umbral = df.loc[df.failed == 0, "score"].quantile(0.95)
df["alerta"] = df["score"] >= umbral

f = df[df.failed == 1]
s = df[df.failed == 0]
n_f = f.VehicleCode.nunique()
detectados = f[(f.dias_hasta_evento <= 30) & f.alerta].VehicleCode.nunique()
primera = f[f.alerta & (f.dias_hasta_evento <= 90)].groupby("VehicleCode").dias_hasta_evento.max()

print(f"Días de sanos con alerta:                  {s.alerta.mean():.1%}")
print(f"Sanos con al menos una alerta:             {s.groupby('VehicleCode').alerta.any().mean():.1%}")
print(f"Fallados alertados en los 30 días previos: {detectados / n_f:.1%}")
print(f"Anticipación mediana:                      {primera.median():.0f} días")
print(f"Días lejanos (>90) de fallados con alerta: {f[f.dias_hasta_evento > 90].alerta.mean():.1%}")

Días de sanos con alerta:                  5.0%
Sanos con al menos una alerta:             57.8%
Fallados alertados en los 30 días previos: 65.9%
Anticipación mediana:                      59 días
Días lejanos (>90) de fallados con alerta: 7.0%


In [ ]:
imp = pd.Series(modelo.booster_.feature_importance("gain"), index=features)
(imp / imp.sum()).sort_values(ascending=False).round(3).head(10)

dist_regen_30d         0.083
aceite_30d             0.080
viajes_30d             0.073
temp_aire_30d          0.068
min_ralenti_30d        0.068
acum_max_30d           0.061
vel_media_30d          0.051
pct_cortos_30d         0.046
ralentis_largos_30d    0.040
pct_motor_frio_30d     0.040
dtype: float64

In [ ]:
def evaluar(cols):
    oof = np.zeros(len(df))
    for tr, te in GroupKFold(n_splits=5).split(df, y, grupos):
        peso = (y.iloc[tr] == 0).sum() / (y.iloc[tr] == 1).sum()
        m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                               min_child_samples=100, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.8, scale_pos_weight=peso, verbose=-1)
        m.fit(df[cols].iloc[tr], y.iloc[tr])
        oof[te] = m.predict_proba(df[cols].iloc[te])[:, 1]
    return round(average_precision_score(y, oof), 4)

sin_regen = [c for c in features if c not in ("regeneraciones_30d", "dist_regen_30d", "tend_dist_regen")]
print("Con todas las features:  ", evaluar(features))
print("Sin las de regeneración: ", evaluar(sin_regen))

Con todas las features:   0.1657
Sin las de regeneración:  0.1561


In [ ]:
orden = df.sort_values(["VehicleCode", "dia"])
df["score_7d"] = orden.groupby("VehicleCode")["score"].transform(lambda s: s.rolling(7, min_periods=1).mean())

def resumen(col, q):
    umbral = df.loc[df.failed == 0, col].quantile(q)
    a = df[col] >= umbral
    f, s = df[df.failed == 1], df[df.failed == 0]
    af, a_s = a[df.failed == 1], a[df.failed == 0]
    return {"score": col,
            "% días sanos alertados": round(1 - q, 3),
            "sanos alertados": round(a_s.groupby(s.VehicleCode).any().mean(), 3),
            "fallados detectados (30d)": round(f[(f.dias_hasta_evento <= 30) & af].VehicleCode.nunique() / f.VehicleCode.nunique(), 3),
            "anticipación mediana": f[af & (f.dias_hasta_evento <= 90)].groupby("VehicleCode").dias_hasta_evento.max().median(),
            "días lejanos alertados": round(af[f.dias_hasta_evento > 90].mean(), 3)}

pd.DataFrame([resumen(c, q) for c in ["score", "score_7d"] for q in [0.95, 0.98, 0.99]])

,score,% días sanos alertados,sanos alertados,fallados detectados (30d),anticipación mediana,días lejanos alertados
0,score,0.05,0.578,0.659,59.0,0.070
1,score,0.02,0.396,0.562,42.0,0.034
2,score,0.01,0.293,0.435,35.0,0.019
3,score_7d,0.05,0.452,0.576,52.0,0.071
4,score_7d,0.02,0.293,0.457,33.0,0.034
5,score_7d,0.01,0.203,0.351,28.5,0.020


In [ ]:
sin_temp = [c for c in features if c != "temp_aire_30d"]
print("Con temp_aire: ", evaluar(features))
print("Sin temp_aire: ", evaluar(sin_temp))

df["mes"] = pd.to_datetime(df["dia"]).dt.month
pd.crosstab(df["mes"], df["target"], normalize="columns").round(3)

Con temp_aire:  0.1657
Sin temp_aire:  0.1601


target,0,1
mes,,
1,0.078,0.076
2,0.075,0.127
3,0.091,0.084
4,0.094,0.057
5,0.104,0.082
6,0.108,0.085
7,0.120,0.095
8,0.081,0.111
9,0.051,0.097


In [ ]:
def mensual(col, q):
    umbral = df.loc[df.failed == 0, col].quantile(q)
    a = df[col] >= umbral
    s = df[df.failed == 0]
    falsa = a[df.failed == 0].groupby([s.VehicleCode, s.edad_dias // 30]).any().mean()
    f = df[(df.failed == 1) & (df.dias_hasta_evento <= 30)]
    det = a[f.index].groupby(f.VehicleCode).any().mean()
    return {"score": col, "umbral q": q,
            "meses sanos con alerta": round(falsa, 3),
            "fallas detectadas (30d previos)": round(det, 3)}

pd.DataFrame([mensual(c, q) for c in ["score", "score_7d"] for q in [0.95, 0.98, 0.99]])

,score,umbral q,meses sanos con alerta,fallas detectadas (30d previos)
0,score,0.95,0.180,0.672
1,score,0.98,0.089,0.572
2,score,0.99,0.055,0.443
3,score_7d,0.95,0.130,0.587
4,score_7d,0.98,0.062,0.465
5,score_7d,0.99,0.036,0.358


In [ ]:
for c in ["pais", "Engine", "ModelSeries"]:
    df[c] = df[c].astype("category")

con_estaticas = features + ["pais", "Engine", "ModelSeries"]
print("Solo comportamiento:      ", evaluar(features))
print("+ país, motor y modelo:   ", evaluar(con_estaticas))

Solo comportamiento:       0.1657
+ país, motor y modelo:    0.189


In [ ]:
for extra in (["pais"], ["Engine"], ["ModelSeries"], ["Engine", "ModelSeries"]):
    print(f"+ {', '.join(extra):22s}", evaluar(features + extra))

+ pais                   0.189
+ Engine                 0.1714
+ ModelSeries            0.1652
+ Engine, ModelSeries    0.1651


In [ ]:
import json, urllib.request

ciudades = pd.read_csv("ciudades_coordenadas.csv").dropna(subset=["lat"])
altitudes = []
for i in range(0, len(ciudades), 50):
    p = ciudades.iloc[i:i + 50]
    url = ("https://api.open-meteo.com/v1/elevation?latitude=" + ",".join(p.lat.round(4).astype(str))
           + "&longitude=" + ",".join(p.lon.round(4).astype(str)))
    with urllib.request.urlopen(url) as r:
        altitudes += json.load(r)["elevation"]

ciudades["altitud_m"] = altitudes
ciudades.to_csv(SALIDAS + "altitud_ciudades.csv", index=False)
ciudades.sort_values("altitud_m", ascending=False).head(8)

,pais,ciudad,ciudad_identificada,lat,lon,fuente_coord,altitud_m
149,PER,HUANCAYO,Huancayo,-12.06866,-75.21027,GeoNames (geonamescache),3255.0
148,PER,CAJAMARCA,Cajamarca,-7.16378,-78.50027,GeoNames (geonamescache),2685.0
135,COL,CHIA,Chía,4.85876,-74.05866,GeoNames (geonamescache),2559.0
131,COL,BOGOTA D.C.,Bogotá,4.71100,-74.07210,manual,2557.0
142,COL,PASTO,Pasto,1.21456,-77.27846,GeoNames (geonamescache),2538.0
147,PER,AREQUIPA,Arequipa,-16.39899,-71.53747,GeoNames (geonamescache),2342.0
138,COL,MANIZALES,Manizales,5.06680,-75.50684,GeoNames (geonamescache),2123.0
139,COL,MEDELLIN,Medellín,6.24500,-75.57151,GeoNames (geonamescache),1473.0


In [ ]:
clave = ciudades["pais"] + "|" + ciudades["ciudad"]
df["altitud_m"] = (df["pais"].astype(str) + "|" + df["ciudad"].astype(str)).map(dict(zip(clave, ciudades["altitud_m"])))
print("Filas sin altitud:", round(df["altitud_m"].isna().mean(), 3))

v = df.groupby("VehicleCode").agg(failed=("failed", "first"), altitud=("altitud_m", "first"))
print(pd.crosstab(pd.cut(v.altitud, [-100, 500, 1500, 2500, 5000]), v.failed, normalize="index").round(2))

print("Solo comportamiento:  ", evaluar(features))
print("+ país:               ", evaluar(features + ["pais"]))
print("+ altitud:            ", evaluar(features + ["altitud_m"]))
print("+ altitud y motor:    ", evaluar(features + ["altitud_m", "Engine"]))

Filas sin altitud: 0.04
failed           0     1
altitud                 
(-100, 500]   0.83  0.17
(500, 1500]   0.64  0.36
(1500, 2500]  0.89  0.11
(2500, 5000]  0.57  0.43
Solo comportamiento:   0.1657
+ país:                0.189
+ altitud:             0.1806
+ altitud y motor:     0.1868


In [ ]:
def evaluar_mono(cols):
    mono = [1 if c == "altitud_m" else 0 for c in cols]
    oof = np.zeros(len(df))
    for tr, te in GroupKFold(n_splits=5).split(df, y, grupos):
        peso = (y.iloc[tr] == 0).sum() / (y.iloc[tr] == 1).sum()
        m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                               min_child_samples=100, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.8, scale_pos_weight=peso,
                               monotone_constraints=mono, verbose=-1)
        m.fit(df[cols].iloc[tr], y.iloc[tr])
        oof[te] = m.predict_proba(df[cols].iloc[te])[:, 1]
    return round(average_precision_score(y, oof), 4)

print("+ altitud libre:       ", evaluar(features + ["altitud_m"]))
print("+ altitud monotónica:  ", evaluar_mono(features + ["altitud_m"]))

+ altitud libre:        0.1806
+ altitud monotónica:   0.1761


In [ ]:
features_v2 = features + ["altitud_m"]
mono_v2 = [1 if c == "altitud_m" else 0 for c in features_v2]
params = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=100,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
              monotone_constraints=mono_v2, verbose=-1)

oof = np.zeros(len(df))
for tr, te in GroupKFold(n_splits=5).split(df, y, grupos):
    m = lgb.LGBMClassifier(scale_pos_weight=(y.iloc[tr] == 0).sum() / (y.iloc[tr] == 1).sum(), **params)
    m.fit(df[features_v2].iloc[tr], y.iloc[tr])
    oof[te] = m.predict_proba(df[features_v2].iloc[te])[:, 1]

df["score_v2"] = oof
orden = df.sort_values(["VehicleCode", "dia"])
df["riesgo"] = orden.groupby("VehicleCode")["score_v2"].transform(lambda s: s.rolling(7, min_periods=1).mean())
print("PR-AUC v2:", round(average_precision_score(y, oof), 4))
print(pd.DataFrame([mensual("riesgo", q) for q in [0.95, 0.98, 0.99]]))

modelo_final = lgb.LGBMClassifier(scale_pos_weight=(y == 0).sum() / (y == 1).sum(), **params)
modelo_final.fit(df[features_v2], y)

PR-AUC v2: 0.1761
    score  umbral q  meses sanos con alerta  fallas detectadas (30d previos)
0  riesgo      0.95                   0.118                            0.635
1  riesgo      0.98                   0.054                            0.491
2  riesgo      0.99                   0.031                            0.384


,learning_rate,0.05
,n_estimators,300
,min_child_samples,100
,subsample,0.8
,subsample_freq,1
,colsample_bytree,0.8
,scale_pos_weight,np.float64(35.356181150550796)
,monotone_constraints,"[0, 0, ...]"
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31


In [ ]:
contrib = modelo_final.booster_.predict(df[features_v2], pred_contrib=True)
shap = pd.DataFrame(contrib[:, :-1], columns=features_v2, index=df.index)

resumen = pd.DataFrame({
    "importancia": shap.abs().mean(),
    "direccion": [np.corrcoef(df[c].fillna(df[c].median()), shap[c])[0, 1] for c in features_v2]
}).sort_values("importancia", ascending=False)
resumen.round(3)

,importancia,direccion
altitud_m,0.667,0.737
min_ralenti_30d,0.513,0.603
aceite_30d,0.348,-0.508
dist_regen_30d,0.314,0.455
vel_media_30d,0.314,0.560
viajes_30d,0.311,-0.817
acum_max_30d,0.266,0.865
regen_completas_30d,0.257,-0.733
pct_lleno_30d,0.236,0.852
acum_media_30d,0.214,-0.353


In [ ]:
umbral = df.loc[df.failed == 0, "riesgo"].quantile(0.98)

def explicar(vehiculo, dia):
    i = df.index[(df.VehicleCode == vehiculo) & (df.dia == dia)][0]
    top = shap.loc[i].sort_values(ascending=False).head(5)
    return pd.DataFrame({"valor": df.loc[i, top.index].round(3), "aporte al riesgo": top.round(3)})

alertas = df[(df.failed == 1) & (df.riesgo >= umbral) & (df.dias_hasta_evento <= 90)]
ej = alertas.sort_values("dias_hasta_evento", ascending=False).iloc[0]
print(f"{ej.VehicleCode}, {ej.dia}: alerta {ej.dias_hasta_evento:.0f} días antes del evento")
explicar(ej.VehicleCode, ej.dia)

In [ ]:
top3 = np.argsort(-contrib[:, :-1], axis=1)[:, :3]
nombres = np.array(features_v2)
df["razon_1"], df["razon_2"], df["razon_3"] = nombres[top3[:, 0]], nombres[top3[:, 1]], nombres[top3[:, 2]]
df["alerta"] = df["riesgo"] >= umbral

salida = df[["VehicleCode", "dia", "failed", "pais", "ciudad", "Engine", "ModelSeries", "altitud_m",
             "dias_hasta_evento", "riesgo", "alerta", "razon_1", "razon_2", "razon_3",
             "pct_lleno_30d", "tend_lleno", "consumo_30d", "pct_cortos_30d", "km_30d"]]
salida.to_parquet(SALIDAS + "scores_dashboard.parquet", index=False)
salida.to_csv(SALIDAS + "scores_dashboard.csv", index=False)
salida.shape

(207921, 19)

In [ ]:
fal = df[df.failed == 1]
fal.groupby("target")[["viajes_30d", "dias_activos_30d", "aceite_30d", "km_30d"]].median()

,viajes_30d,dias_activos_30d,aceite_30d,km_30d
target,,,,
0,153.0,27.0,61.0,2032.0
1,124.0,25.0,57.0,1703.0


In [ ]:
STATIC = DATOS + "Static/"
est = pd.concat([
    pd.read_csv(STATIC + "StaticInformation_FailedVins_v2_20260924_144239.csv", encoding="utf-8-sig"),
    pd.read_csv(STATIC + "StaticInformation_NotFailedVins_v2_20260924_144209.csv", encoding="utf-8-sig"),
]).drop_duplicates("VehicleCode")

df["dias_venta"] = df["VehicleCode"].map(dict(zip(est.VehicleCode, est.daysUntilSale)))
df["antes_venta"] = df["edad_dias"] < df["dias_venta"]

print("Filas antes de la venta:", round(df["antes_venta"].mean(), 3))
print("Vehículos sin fecha de venta:", df.loc[df.dias_venta.isna(), "VehicleCode"].nunique())
df.groupby(["failed", "antes_venta"])[["pct_cortos_30d", "km_30d", "target"]].mean().round(3)